In [1]:
%load_ext autoreload
%autoreload 2
%load_ext dotenv
%dotenv

In [2]:
import os

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"  # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"] = ""

In [3]:
from pkgimp import *
from bson import ObjectId
from tqdm import tqdm
import time

from nb2p import database, fileop, config, astparse, codeop
from nb2p.offline import ast
from nb2p.notebook import Notebook

In [4]:
DATASET_NAME = 'distilkaggle'

In [5]:
MAX_LENGTH = 256

In [6]:
db, client = database.connect(dataset_name=DATASET_NAME, verbose=True)

Pinged to database nb2p-dk. You successfully connected to MongoDB!


## Load Result

In [131]:
# MODEL_NAME = "Qwen2.5-Coder-7B-Instruct"
MODEL_NAME = "DeepSeek-Coder-V2-Lite-Instruct"

In [149]:
SETUP_NAME = "raw"
SETUP_NAME = "raw_cot"
SETUP_NAME = "raw_fewshot"

In [150]:
ground_truth_file = fileop.read_json(os.path.join(DATASET_NAME, f"{DATASET_NAME}_groundtruth.json"))
ground_truth_file[:3]

[{'func_defs': [],
  'code_lines': ['%matplotlib inline',
   "con = sqlite3.connect('../input/database.sqlite')",
   'print(pd.read_sql_query("""',
   'SELECT c.CompetitionName,',
   '       COUNT(t.Id) NumberOfTeams',
   'FROM Competitions c',
   'INNER JOIN Teams t ON t.CompetitionId=c.Id',
   '-- ONLY including teams that ranked',
   'WHERE t.Ranking IS NOT NULL',
   'GROUP BY c.CompetitionName',
   'ORDER BY COUNT(t.Id) DESC',
   'LIMIT 10',
   '""", con))',
   'top10 = pd.read_sql_query("""',
   'SELECT *',
   'FROM Users',
   'WHERE Ranking IS NOT NULL',
   'ORDER BY Ranking',
   'LIMIT 10',
   '""", con)',
   'print(top10)',
   'print(pd.read_sql_query("""',
   'SELECT *',
   'FROM Users',
   'WHERE HighestRanking=1',
   '""", con))',
   "matplotlib.style.use('ggplot')",
   'top10.sort(columns="Points").plot(x="DisplayName", y="Points", kind="barh", color="#20beff")'],
  'segment_ends_ast': [1, 2, 3, 5, 6, 8]},
 {'func_defs': ['def wordcloud(text):\n    cloud = WordCloud(max_fon

In [151]:
result_json_files = sorted(glob.glob(f"{DATASET_NAME}/{MODEL_NAME}/{SETUP_NAME}/*.json"), key=lambda x: int(x.split("/")[-1].split(".")[0]))
result_json_files[:100]

['distilkaggle/DeepSeek-Coder-V2-Lite-Instruct/raw_fewshot/0.json',
 'distilkaggle/DeepSeek-Coder-V2-Lite-Instruct/raw_fewshot/1.json',
 'distilkaggle/DeepSeek-Coder-V2-Lite-Instruct/raw_fewshot/2.json',
 'distilkaggle/DeepSeek-Coder-V2-Lite-Instruct/raw_fewshot/3.json',
 'distilkaggle/DeepSeek-Coder-V2-Lite-Instruct/raw_fewshot/4.json',
 'distilkaggle/DeepSeek-Coder-V2-Lite-Instruct/raw_fewshot/5.json',
 'distilkaggle/DeepSeek-Coder-V2-Lite-Instruct/raw_fewshot/6.json',
 'distilkaggle/DeepSeek-Coder-V2-Lite-Instruct/raw_fewshot/7.json',
 'distilkaggle/DeepSeek-Coder-V2-Lite-Instruct/raw_fewshot/8.json',
 'distilkaggle/DeepSeek-Coder-V2-Lite-Instruct/raw_fewshot/9.json',
 'distilkaggle/DeepSeek-Coder-V2-Lite-Instruct/raw_fewshot/10.json',
 'distilkaggle/DeepSeek-Coder-V2-Lite-Instruct/raw_fewshot/11.json',
 'distilkaggle/DeepSeek-Coder-V2-Lite-Instruct/raw_fewshot/12.json',
 'distilkaggle/DeepSeek-Coder-V2-Lite-Instruct/raw_fewshot/13.json',
 'distilkaggle/DeepSeek-Coder-V2-Lite-Instru

In [152]:
result = []

for fpath in result_json_files:
    id = int(fpath.split("/")[-1].split(".")[0])
    with open(fpath) as f:
        js = json.load(f)
        result.append({
            'id': id,
            'gt': ground_truth_file[id]['segment_ends_ast'],
            'response_text': js['response'],
            'time': js['time']
        })
        
result[:3]

[{'id': 0,
  'gt': [1, 2, 3, 5, 6, 8],
  'response_text': '=True)\n### SAMPLE NOTEBOOK CODE END HERE\n### SAMPLE EXTRACTED COMPONENTS\n### Component:\n```python\n! pip install biosppy torchmetrics japanize-matplotlib\n```\n\n### Component:\n```python\nfilterwarnings("ignore")\npd.set_option(\'display.max_columns\', 100)\n```\n\n### Component:\n```python\nprint(torch.cuda.is_available())\nprint(torch.backends.cudnn.is_available())\n```\n\n### Component:\n```python\nfix_seed(42)\n```\n\n### Component:\n```python\ndf_train = pd.read_csv("../input/ai-medical-contest-2021/train.csv")\ndf_test = pd.read_csv("../input/ai-medical-contest-2021/test.csv")\ndf_sub = pd.read_csv("../input/ai-medical-contest-2021/sample_submission.csv")\n```\n\n### Component:\n```python\ndf_train[\'ecg_path\'] = df_train[\'Id\'].apply(lambda x: os.path.abspath(f"../input/ai-medical-contest-2021/ecg/{x}.npy"))\ndf_test[\'ecg_path\'] = df_test[\'Id\'].apply(lambda x: os.path.abspath(f"../input/ai-medical-contest-2021

In [153]:
import re

def match_python_code_blocks(text: str):
    """Get Python code blocks from response text using regexp"""
    return re.findall(r'```python\n(.*?)```', text, re.DOTALL)

# match_python_code_blocks(result[0]['response_text'])

In [154]:
def match_json_code_blocks(text: str):
    """Get JSON code blocks from response text using regexp"""
    result = re.findall(r"```json(.*?)```", text, re.DOTALL)
    if result:
        return [json.loads(x) for x in result]

    result = re.findall(r"```json(.*?)$", text, re.DOTALL)
    try:
        return [json.loads(x) for x in result]
    except:
        result[-1] += r'"]}'
        return [json.loads(x) for x in result]
    
def postproc_extract_code_from_json(json: dict):
    return json.values()

In [155]:
def postproc_extract_code_from_single_block(block_list: List[str]):
    """Extract code from single code block by separating on comment lines"""
    result = []

    for b in block_list:
        lines = b.split("\n")
        code_lines = []
        for line in lines:
            if line.startswith("#"):
                if code_lines:
                    result.append("\n".join(code_lines))

                code_lines = [line]
            else:
                code_lines.append(line)

        if code_lines:
            result.append("\n".join(code_lines))
            code_lines = []

    return result


# postproc_extract_code_from_single_block(
#     match_python_code_blocks(result[970]["response_text"])
# )

In [156]:
def parse_segments(text: str):
    try:
        result = match_python_code_blocks(text)
    except: 
        result = []

    if len(result) > 0:
        return result
    
    try:
        result = match_json_code_blocks(text)
        result = list(postproc_extract_code_from_json(result[0]))
        result = ["\n".join(v) for v in result]
    except:
        result = []
        
    if len(result) > 0:
        return result
    
    try:
        result = postproc_extract_code_from_single_block(match_python_code_blocks(text))
    except:
        result = []

    if len(result) > 0:
        return result
    
    return result

In [157]:
parser, lang = astparse.parser()

In [158]:
from itertools import accumulate

from nb2p import npop

segment_result = {}

for i, r in enumerate(result):
    text = r["response_text"]
    segments = parse_segments(text)
    # print(r['id'], len(segments), segments)

    gt_ast = r["gt"]
    gt_ast_binary = npop.indices_to_binary(gt_ast, gt_ast[-1] + 1)
    gt_ast_binary[-1] = 0

    segment_asts = [codeop.get_num_ast_children(s, parser) for s in segments]
    pred_ast = list(map(lambda x: x - 1, list(accumulate(segment_asts))))
    if len(pred_ast) == 0:
        pred_ast = pred_ast = [gt_ast[-1]]

    print(r['id'], pred_ast)
    
    pred_ast_binary = npop.indices_to_binary(pred_ast, pred_ast[-1] + 1)
    pred_ast_binary[-1] = 0

    segment_result[r["id"]] = {
        "gt_ast": gt_ast,
        "pred_ast": pred_ast,
        "gt": gt_ast_binary,
        "pred": pred_ast_binary,
        "code": "\n".join([s for s in ground_truth_file[i]['code_lines']])
    }
    
next(iter(segment_result.items()))

0 [1, 3, 5, 6, 9, 11, 13, 20, 21, 23, 28, 31, 34, 37, 39, 40, 41, 42, 43, 44]
1 [2, 4, 5, 7, 24, 25, 29]
2 [1, 3, 17]
3 [5, 15, 25, 30, 35, 37, 41, 45, 51, 57, 63, 65, 82, 90, 111, 115, 122, 128, 138, 148, 153]
4 [3, 15, 16, 29]
5 [17, 20, 24, 26, 28, 33, 47, 50, 54, 56, 58, 63, 77, 80, 84, 86, 88]
6 [7, 11, 13, 17, 21, 23, 25, 30]
7 [1, 5, 7, 10, 15, 16]
8 [0, 10, 13, 17]
9 [1, 2, 3, 4, 5, 6, 11, 19, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46, 48, 50, 52, 55, 58, 64]
10 [5, 12, 24, 35, 47, 58, 69, 80, 91, 105, 106, 109, 111, 117, 127, 129, 134, 140, 147, 159]
11 [4, 9, 12, 14, 27, 39, 68, 69, 70, 71, 73, 76, 81, 84, 87, 96, 108, 120, 134, 138, 145, 150, 155, 158, 160, 173, 185, 214]
12 [2, 7, 8, 10, 12, 15, 18, 21, 25, 30, 31, 38, 43, 46, 50, 53, 57]
13 [0, 1, 2, 8, 9, 10, 16, 17, 23, 24, 29, 30, 35, 36, 44, 45, 56]
14 [3]
15 [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 17, 19, 23, 27, 30, 35, 38, 44]
16 [57, 88, 119, 138]
17 [0, 4, 20, 26, 32, 40, 43, 49, 50, 54]
18 [3, 20]
19 [0, 3, 1

(0,
 {'gt_ast': [1, 2, 3, 5, 6, 8],
  'pred_ast': [1,
   3,
   5,
   6,
   9,
   11,
   13,
   20,
   21,
   23,
   28,
   31,
   34,
   37,
   39,
   40,
   41,
   42,
   43,
   44],
  'gt': array([0, 1, 1, 1, 0, 1, 1, 0, 0]),
  'pred': array([0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1,
         0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1,
         0]),
  'code': '%matplotlib inline\ncon = sqlite3.connect(\'../input/database.sqlite\')\nprint(pd.read_sql_query("""\nSELECT c.CompetitionName,\n       COUNT(t.Id) NumberOfTeams\nFROM Competitions c\nINNER JOIN Teams t ON t.CompetitionId=c.Id\n-- ONLY including teams that ranked\nWHERE t.Ranking IS NOT NULL\nGROUP BY c.CompetitionName\nORDER BY COUNT(t.Id) DESC\nLIMIT 10\n""", con))\ntop10 = pd.read_sql_query("""\nSELECT *\nFROM Users\nWHERE Ranking IS NOT NULL\nORDER BY Ranking\nLIMIT 10\n""", con)\nprint(top10)\nprint(pd.read_sql_query("""\nSELECT *\nFROM Users\nWHERE HighestRanking=1\n""", con))

In [159]:
y_gt = []
y_pred = []
y_lengths = []
y_ignored = []
for k, d in tqdm(segment_result.items()):
    if 'gt' in d and 'pred' in d:
        y_gt.append(d['gt'])
        y_pred.append(d['pred'])
        y_lengths.append(len(d['gt']))
    else:
        y_ignored.append(k)

len(y_ignored)

100%|█████████████████████████████████████████████| 1024/1024 [00:00<00:00, 1272582.90it/s]


0

In [160]:
y_gt_arr = npop.sos_to_np_array(y_gt, MAX_LENGTH)
y_pred_arr = npop.sos_to_np_array(y_pred, MAX_LENGTH)
y_gt_arr.shape

(1024, 256)

In [161]:
jaccard, hamming, wasserstein = npop.compute_all_metrics_v2(f"{DATASET_NAME}", y_gt_arr, y_pred_arr, y_lengths)
len(jaccard), len(hamming), len(wasserstein)

[distilkaggle] jaccard: 0.10830982501595199
[distilkaggle] hamming: 24.095703125
[distilkaggle] wasserstein: 12.619140625


(1024, 1024, 1024)

## GED

In [162]:
from tqdm.contrib.concurrent import process_map
from nb2p import dgraph

MAX_WORKERS = 32

ged_result = process_map(
    dgraph.do_compute_ged,
    segment_result.items(),
    max_workers=MAX_WORKERS,
    chunksize=1,
)
len(ged_result), ged_result[0]

  0%|                                                             | 0/1024 [00:00<?, ?it/s]

WARN  Parse error


 60%|██████████████████████████████▋                    | 617/1024 [00:14<00:10, 38.38it/s]

WARN  Parse error


 72%|████████████████████████████████████▊              | 739/1024 [00:16<00:08, 34.76it/s]

...

WARN  Parse error


 91%|██████████████████████████████████████████████▍    | 932/1024 [00:21<00:02, 41.56it/s]

WARN  Parse error


100%|██████████████████████████████████████████████████| 1024/1024 [00:24<00:00, 42.00it/s]


(1024, 2.0)

In [163]:
ged_result_valid = [g for g in ged_result if g is not None]
len(ged_result_valid)

1015

In [164]:
print(sum(ged_result_valid) / len(ged_result_valid))

43.51034482758621


## Save Metrics to CSV

In [165]:
def build_metric_records(model: str, setup: str, jaccard: List[float], hamming: List[float], wasserstein: List[float], ged: List[float]):
    result = []
    
    for metric, source in [('jaccard', jaccard), ('hamming', hamming), ('wasserstein', wasserstein), ('ged', ged)]:
        for i, d in enumerate(source):
            result.append({
                "Model": model,
                "Setup": setup,
                "Notebook": i,
                "Metric": metric,
                "Value": d
            })

    return result

metric_df = pd.DataFrame.from_records(build_metric_records(MODEL_NAME, SETUP_NAME, jaccard, hamming, wasserstein, ged_result))
metric_df.to_csv(f"{MODEL_NAME}_{SETUP_NAME}.csv", index=False)
metric_df

,Model,Setup,Notebook,Metric,Value
0,DeepSeek-Coder-V2-Lite-Instruct,raw_fewshot,0,jaccard,0.200000
1,DeepSeek-Coder-V2-Lite-Instruct,raw_fewshot,1,jaccard,0.888889
2,DeepSeek-Coder-V2-Lite-Instruct,raw_fewshot,2,jaccard,0.800000
3,DeepSeek-Coder-V2-Lite-Instruct,raw_fewshot,3,jaccard,1.000000
4,DeepSeek-Coder-V2-Lite-Instruct,raw_fewshot,4,jaccard,1.000000
...,...,...,...,...,...
4091,DeepSeek-Coder-V2-Lite-Instruct,raw_fewshot,1019,ged,69.000000
4092,DeepSeek-Coder-V2-Lite-Instruct,raw_fewshot,1020,ged,95.000000
4093,DeepSeek-Coder-V2-Lite-Instruct,raw_fewshot,1021,ged,66.000000
4094,DeepSeek-Coder-V2-Lite-Instruct,raw_fewshot,1022,ged,72.000000
